In [ ]:
!pip install ipywidgets

In [ ]:
from IPython.display import display
from ipywidgets import HTML

# Machine Learning - Aufgabenblatt 3

## Datensatz

In diesem Aufgabenblatt verwenden wir den Iris-Datensatz.

Der Iris-Datensatz wird in vielen Beispielen und Büchern über Classification in Machine Learning verwendet.
Sie finden also online zusätzliche Informationen und Lösungs-Ansätze zu diesem Datensatz.
Der Datensatz ist sogar in `sklearn` integriert, sprich wir können den Iris-Datensatz direkt über `sklearn.datasets`  laden.

Der Iris-Datensatz ist hier beschrieben: https://en.wikipedia.org/wiki/Iris_flower_data_set

| Feature            | Descriptiopn                                                                  |
|--------------------|-------------------------------------------------------------------------------|
| specie             | Spezie der Blume                                                              |
| sepal length (cm)  | Länge des Kelchblattes der Blume (in Zentimeter).                             |
| sepal width (cm)   | Breite des Kelchblattes der Blume (in Zentimeter).                            |
| petal length (cm)  | Länge des Kronblattes der Blume (in Zentimeter).                              |
| petal width (cm)   | Breite des Kelchblattes der Blume (in Zentimeter).                            |

## Ziel

Das Ziel ist es anhand der Features vorhersagen zu können, um welche Blumenart es sich handelt.
Wir sagen also basierend auf Inputs (`Features`) ein Output (`Label`, auch `Klasse`) voraus - wir machen also eine `Classification`.

![Ziel des Aufgabenblattes](./img/goal.png)

## Setup

Im Setup sind bereits notwendige Schritte implementiert, die Sie im Aufgabenblatt 2 selbst implementierten.
**Nach Aufgabenblatt 2 sollten die Schritte klar sein!**

Wir laden hier die Daten über `sklearn.datasets`, und teilen anschliessend die Features in `X` und Zielvariable in `y` auf.

In [ ]:
from sklearn import datasets

# Laden des Iris-Datensatzes über sklearn.datasets
iris = datasets.load_iris(as_frame=True)

# Aufteilen der Daten in Features und Zielvariable
X = iris.data
y = iris.target

# iris.target ist als Nummer abgespeichert (0, 1 oder 2), hier schlüsseln wir diese Codierung in die Spezien-Namen auf (target_names).
y = y.apply(lambda key: iris.target_names[key]).rename('specie')

display(HTML('X:'))
display(X.head())
display(HTML('y:'))
display(y.to_frame().head())

Analog zum Aufgabenblatt 2 machen wir hier ein `Test-Set`, `Validation-Set` und `Train-Set`, welche wir in den Aufgaben verwenden werden.

In [ ]:
from sklearn.model_selection import train_test_split

# Split in Test-Set and Data-Set
X_data, X_test, y_data, y_test = train_test_split(X, y, random_state=2, stratify=y)
# Split Data-Set in Train-Set and Validation-Set
X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, random_state=2, stratify=y_data)

print("Test-Set", X_test.shape)
print("Data-Set", X_data.shape)
print("Train-Set", X_train.shape)
print("Validation-Set", X_val.shape)

Für `seaborn` in der Datenanalyse (Aufgabe 1) setzen wir `X_data` und `y_data` zu einem DataFrame `df_data` zusammen.

In [ ]:
import pandas as pd

df_data = pd.concat([X_data, y_data.to_frame()], axis=1)

display(df_data)

Folgende Funktion müssen Sie in den Aufgaben verwenden. Die Umsetzung (der Code) der Funktion **muss nicht verstanden** werden. 
Die Funktion hilft die `Decision Region` eines Classifiers zu plotten. Wir verwenden die Funktion in späteren Aufgaben, um die Classifiers besser zu verstehen. 

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
from matplotlib.colors import ListedColormap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_decision_regions(clf, data: pd.DataFrame, x: str, y: str, colors = None, grid_resolution=1000):
    """
    Funktion zum Plotten der Decision Region eines Classification Modells.
    Verwendet nun das moderne DecisionBoundaryDisplay von scikit-learn.

    :param clf: Classification Modell
    :param data: Daten als DataFrame
    :param x: Spaltenname der X-Achse
    :param y: Spaltenname der Y-Achse
    :param colors: Farben als Dictionary (optional)
    :param grid_resolution: Auflösung des Decision Region Plots (grid_resolution).
    """

    cmap = None
    if colors is not None:
        # Force the colors into the exact order scikit-learn expects
        ordered_colors = [colors[class_name] for class_name in clf.classes_]
        cmap = ListedColormap(ordered_colors)

    X_plot = data[[x, y]]

    DecisionBoundaryDisplay.from_estimator(
        estimator=clf,
        X=X_plot,
        response_method="predict",
        grid_resolution=grid_resolution,
        cmap=cmap,
        alpha=0.25,
        xlabel=x,
        ylabel=y,
        ax=plt.gca() # Ensures it draws on the currently active matplotlib figure
    )

## Aufgabe 1 - Daten Analyse

Wir möchten Anhand unserer Features (wie `sepal length (cm)`) die Blumenspezie (`specie`) vorhersagen können.

In Aufgabe 2 werden wir dafür ein Modell erstellen. In Aufgabe 1 werden wir erst einmal die Daten mit ein paar Plots genauer anschauen.

#### Aufgabe 1.1 - Histogram der Zielvariable

Es macht immer Sinn, sich die **Verteilung der Zielvariable** anzuschauen.
Bei einer `Classification` kann man einfach die Anzahl Datenpunkte pro `Klasse` ausgeben.

1. Erstellen Sie ein Plot der Zielvariable `specie` mittels `sns.countplot` (oder `sns.histplot`)
2. Interpretieren Sie den Plot.

In [ ]:
# TODO

### Aufgabe 1.2 - `sns.stripplot`, `sns.violinplot`

1. Erstellen Sie einen `scatter plot` vom `Data-Set` (`df_data`) für das Feature `sepal width (cm)` und der Klasse `specie` mittels `sns.stripplot`. Der Plot heisst `stripplot` weil es die Datenpunkte vom `scatter plot` in einem Streifen (engl. `stirp`) anordnet. Würden wir das nicht machen, könnten wir die Menge der Punkte schlechter beurteilen.
2. Wiederholen Sie Schritt 1 für alle Features. Dies kann man mittels `for` Schleife (einfacher) oder mit einem `sns.PairGrid` (schwieriger) machen. Die Spalten des DataFrames kann man mit `df_data.columns` auslesen.
3. Interpretieren Sie den erstellten Plot.
4. Wiederholen Sie Schritt 2 mit dem `sns.violinplot`.
5. (Extra) Es gibt noch weitere interessante Plots wie der `sns.swarmplot` oder den `sns.boxplot`.
6. (Extra) Was ist der Unterschied von `sns.stripplot` und `sns.violinplot`?

#### Hilfreiche Links

* `sns.stripplot`: https://seaborn.pydata.org/generated/seaborn.stripplot.html
* `sns.violinplot`: https://seaborn.pydata.org/generated/seaborn.violinplot.html
* A complete guide to plotting categorical variables: https://towardsdatascience.com/a-complete-guide-to-plotting-categorical-variables-with-seaborn-bfe54db66bec

In [ ]:
# TODO

### Schlusswort Aufgabe 1

Aufgabe 1 gibt Ihnen eine Idee der `Datenanalyse` für die `Classification`.
Wir haben ein verschiedene hilfreiche Plots, wie den `countplot`, den `stripplot` und den `violinplot`  gesehen.
Dies ist aber bei weitem nicht alles was man tun kann.
Wir haben in Aufgabe 1.2 alle Features unabhängig von einander angeschaut, man könnte die Feature Interaktionen bereits in der Datenanalyse untersuchen.

## Aufgabe 2 - Logistic Regression

In dieser Aufgabe erstellen wir unser erstes `Classification` Modell, ein `Logistic Regression` Modell.
Die `LogisticRegression` ist ein `Classification`-Modell und **kein** `Regression`-Modell, trotz des irreführenden Namens!

### Aufgabe 2.1 - Plot Features - `sns.scatterplot(..., hue='col')`

Wieder aus **didaktischen Gründen** (analog zu Aufgabenblatt 2) verwenden wir hier **lediglich 2 Features**, so können wir unsere Datenpunkte und trainierten Modelle 2-dimensional visualisieren.
Die X-Achse und Y-Achse sind jeweils die Features und die Farbe der Punkte (`hue`) ist das Label der Zielvariable.

Wir verwenden hier die Features `petal length (cm)` und `petal width (cm)`.

1. Visualisieren Sie die beiden Features `petal length (cm)` und `petal width (cm)` vom `Data-Set` (`df_data`) mittels `sns.scatterplot` und färben sie die Punkte mit dem `hue` Parameter nach der Zielvariable `specie` ein.
2. In Aufgabe 2.2 trainieren wir eine `LogisticRegression`, was genau wird dieses Modell schlussendlich machen? Überlegen Sie sich es am Plot aus Schritt 1.
3. (Extra) Was ist bei den Überlegungen aus Schritt 2 der Unterschied zur Regression?

In [ ]:
# TODO

### Aufgabe 2.2 - `sklearn.linear_model.LogisticRegression`

Analog zur `LinearRegression` vom Aufgabenblatt 2, trainieren wir hier die `LogisticRegression`.

1. Erstellen Sie eine `LogisticRegression` und trainieren Sie diese auf den Features `petal length (cm)` und `petal width (cm)` vom `Train-Set` (`X_train`, `y_train`).
2. Predicten (`clf.predict`) Sie die Vorhersagen auf dem `Validation-Set` (`X_val`). Hier müssen Sie wieder die entsprechenden Features selektieren.
3. Messen Sie die Genauigkeit (englisch Accuracy) mittels `sklearn.metrics.accuracy_score` von `y_val_hat` unseren Vorhersagen und `y_val` den tatsächlichen `Klassen`.
4. Visualisieren Sie die `Decision Boundary` von unserem Modell mit der Hilfe von der oben definierten `plot_decision_regions` Funktion. Visualisieren Sie zusätzlich die Datenpunkte mit `sns.scatterplot`. Was zeigt dieser Plot?
5. (Extra) In `sklearn.metrics` gibt es weitere Metriken, manche haben wir im Theorie Teil kennengelernt, wie den F1-Score (`f1_score`). Schauen Sie sich den `classification_report` an (Sammlung von solchen Metriken) und interpretieren Sie diese Metriken.

In [ ]:
# TODO

### Aufgabe 2.3 - Confusion Matrix

In Aufgabe 2.2 haben wir ein Modell trainiert und die Genauigkeit gemessen.
Oft ist es aber interessant, wo genau das Modell die Fehler macht.
In der `Classification` bedeutet ein Fehler nämlich immer, dass wir einen Datenpunkt einer falschen `Klasse` zuordnen.
Dafür haben wir im Theorie Teil die `Confusion Matrix` kennengelernt.

Die Berechnung `Confusion Matrix` müssen wir nicht selbst programmieren, `sklearn` bietet bereits eine Implementierung dafür.

1. Erstellen Sie mittels `sklearn.metrics.confusion_matrix` den Vorhersagen `y_val_hat` von Aufgabe 2.2 und den richtigen Labels `y_val` die `Confusion Matrix` auf dem `Validation-Set` und interpretieren Sie die `Confusion Matrix`.

In [ ]:
# TODO

### (Extra) Aufgabe 2.4 - Was wurde gelernt?

Die `Logistische Regression` hat folgende Form:

$
\begin{aligned}
    \phi(x^{(i)}\beta) = \phi(\beta_0 + x_1 \beta_1 + \cdots + x_p \beta_p)
\end{aligned}
$

Dies ist einfach die `Lineare Regression` mit der Sigma-Funktion $\phi$ als `Linker-Funktion`.

Da wir nur zur Zeit nur zwei Features haben, vereinfacht sich das zu:

$
\begin{aligned}
    \phi(x^{(i)}\beta) = \phi(\beta_0 + x_1 \beta_1 + x_2 \beta_2)
\end{aligned}
$

Analog zum Aufgabenblatt 2 können wir auch nachschauen, welche $\beta$s (Gewichte) gelernt wurden.

```python
print("beta0", clf.intercept_)
print("beta1", clf.coef_[:, 0])
print("beta2", clf.coef_[:, 1])
```

1. Wenn wir die Parameter anschauen, sehen wir, es wurden jeweils drei verschiedene $\beta_0$, $\beta_1$ und $\beta_2$ gelernt! Warum haben wir insgesamt 9 und nicht 3 Parameter? Erkläre.

In [ ]:
# TODO

### Schlusswort Aufgabe 2

In Aufgabe 2 haben wir unser erstes `Classification` Modell, eine `Logistische Regression`, angewandt.
In den nächsten Aufgaben werden wir weitere `Classification` Modelle anwenden.

Für die Auswertung unseres Modelles nutzten wir `Classification` Metriken, wie die `Genauigkeit` (Accuracy) 
und die `Confusion Matrix`.
Diese Auswertungen können wir genau gleich wieder bei anderen Modellen anwenden.

Der Iris-Datensatz ist bereits **linear gut separierbar**, mit nur 2 Featuren und ohne `Feature Engineering` und ohne nicht-lineare Modelle.
Obwohl der Datensatz (zu) einfach ist bleiben wir auch für die weiteren Aufgaben auf dem Iris-Datensatz.
Es geht in den weiteren Aufgaben mehr um die Visualisierungen der Modelle, sowie dem Verständnis wie sie sich unterscheiden.

## Aufgabe 3 - Decision Tree

In dieser Aufgabe erstellen wir ein `Decision Tree` Modell für die `Classification`.
`Decision Tree` Modelle können für `Classification` und `Regression` eingesetz werden.

### Aufgabe 3.1 - `sklearn.tree.DecisionTreeClassifier`

1. Erstellen Sie eine `DecisionTreeClassifier(random_state=42)` und trainieren Sie diese auf den Features `petal length (cm)` und `petal width (cm)` vom `Train-Set` (`X_train`, `y_train`).
2. Predicten (`clf.predict`) Sie die Vorhersagen auf dem `Validation-Set` (`X_val`). Hier müssen Sie wieder die entsprechenden Features selektieren.
3. Messen Sie die Genauigkeit (englisch Accuracy) mittels `sklearn.metrics.accuracy_score` von `y_val_hat` unseren Vorhersagen und `y_val` den tatsächlichen `Klassen`.
4. Visualisieren Sie die `Decision Boundary` von unserem Modell mit der Hilfe von der oben definierten `plot_decision_regions` Funktion.

In [ ]:
# TODO

### Aufgabe 3.2

Wir haben in Aufgabe 2.2 und Aufgabe 3.1 die `Decision Regions` vom Logistischen Regression Modell und vom Decision Tree Modell visualisiert. Das sollte in etwa so aussehen (mit `random_state=42`):

Logistische Regression     |  Decision Tree
:-------------------------:|:-------------------------:
![Logistische Regression oder SVM?](img/lr.png) |  ![Logistische Regression oder SVM?](img/tree.png)

1. Erklären Sie den Unterschied der `Decision Boundaries` mit Ihrem Wissen darüber wie diese Modelle funktionieren und trainiert werden. Woran man das zugrundeliegende Modell anhand der `Decision Regions` erkennen?


In [ ]:
# TODO

### Aufgabe 3.3 - Decision Tree - Alle Features und Hyper-Parameter Optimization

Nun möchten wir alle Features verwenden und die Hyper-Parameter vom Modell mittels `Hyper-Parameter Optimierung` finden.

Dazu können wir unterschiedliche Such-Strategien verwenden:
* `Grid Search`: Systematisch ein Grid absuchen
* `Randomized Search`: `n` Mal zufälle Parameter in einem fixen Bereich ausprobieren

Wir verwenden hier `Randomized Search`, da es in der Praxis üblicherweise besser funktioniert.
`sklearn` bietet bereits eine Implementierung mit `Cross Validation` für stabilere Aussagen für die einzelnen Parameter bereit namens `RandomizedSearchCV`.

1. Erstellen Sie einen fixen Bereich für die `max_depth` und `min_samples_split` Parameter. RandomizedSearch wird innerhalb dieser Bereichen suchen (zufällige Zahlen ziehen). Beispiel-Code:
```python
param_dist = dict(
    max_depth=randint(1, 50),
    min_samples_split=randint(2, 20)
)
```
2. Erstellen Sie eine `RandomizedSearchCV` mit einem `DecisionTreeClassifier(random_state=42)` Modell und dem in Schritt 1 erstellten fixen Bereich. Wählen Sie eine sinnvolle Anzahl an Iterationen (z.B. `n_iter=25`).
3. Trainieren `fit` Sie die `RandomizedSearchCV` auf dem `Data-Set` (`X_data` `y_data`).
4. Warum nehmen wir in Schritt 3 das `Data-Set` und nicht das `Train-Set`?
5. Welches Hyper-Parameter wurden gefunden? Verwenden Sie dazu `rscv.best_params_`.

In [ ]:
# TODO

### Schlusswort Aufgabe 3

In Aufgabe 3 haben wir ein `Decision Tree` `Classification` Modell erstellt und trainiert.
Anschliessend haben wird mittels `Hyper-Parameter Optimierung` gute Werte für die Hyper-Parameter `max_depth` und `min_samples_split` gefunden.
Auf diesem Toy-Datensatz bringt die Hyper-Parameter Optimierung wenig Verbesserung, in der Praxis ist die Hyper-Parameter Optimierung oft ein wichtiger Schritt, um die Performanz eines Modelles um ein paar Prozentpunkte zu verbessern.

## (Extra) Aufgabe 4 - Weitere Modelle

Analog zur Aufgabe 2.2 (Logistische Regression) und Aufgabe 3.1 (Support Vector Machine), trainieren wir hier andere Classification-Modelle, um ein Gefühl für den Unterschied zu bekommen.

### (Extra) Aufgabe 4.1 - RandomForestClassifier

1. Erstellen Sie eine `sklearn.ensemble.RandomForestClassifier` und trainieren Sie diese auf den Features `petal length (cm)` und `petal width (cm)` vom `Train-Set` (`X_train`, `y_train`).
2. Predicten (`clf.predict`) Sie die Vorhersagen auf dem `Validation-Set` (`X_val`). Hier müssen Sie wieder die entsprechenden Features selektieren.
3. Messen Sie die Genauigkeit (englisch Accuracy) mittels `sklearn.metrics.accuracy_score` von unseren Vorhersagen `y_val_hat` und den tatsächlichen Labels `y_val`.
4. Visualisieren Sie die decision boundary von unserem Modell mit der Hilfe von der oben definierten `plot_decision_regions` Funktion. Was zeigt dieser Plot?

In [ ]:
# TODO

### (Extra) Aufgabe 4.2 - KNN (K-Nearest-Neighbors)

1. Erstellen Sie eine `sklearn.neighbors.KNeighborsClassifier` und trainieren Sie diese auf den Features `petal length (cm)` und `petal width (cm)` vom `Train-Set` (`X_train`, `y_train`).
2. Predicten (`clf.predict`) Sie die Vorhersagen auf dem `Validation-Set` (`X_val`). Hier müssen Sie wieder die entsprechenden Features selektieren.
3. Messen Sie die Genauigkeit (englisch Accuracy) mittels `sklearn.metrics.accuracy_score` von unseren Vorhersagen `y_val_hat` und den tatsächlichen Labels `y_val`.
4. Visualisieren Sie die decision boundary von unserem Modell mit der Hilfe von der oben definierten `plot_decision_regions` Funktion. Was zeigt dieser Plot?

In [ ]:
# TODO

### (Extra) Aufgabe 4.3 - Support Vector Machine

#### (Extra) Aufgabe 4.3.1

1. Die Support Vector Machine (SVM) ist ein weiteres Machine Learning Modell mit historischer Relevanz. Die SVM ist eher von geometrischen Überlegungen motiviert: Es findet die Support Vectoren, die eine Decision Boundary beschreiben, welche die Datenklassen separaiert und maximalen Abstand zu den Datenpunkten aufweist. Erarbeiten Sie sich selbstständig ein Verständnis über die Support Vector Machine (z.B. https://de.wikipedia.org/wiki/Support_Vector_Machine). Formulieren Sie die Kern-Unterschiede zur Logistischen Regression.



### (Extra) Aufgabe 4.3.2

1. Erstellen Sie eine `sklearn.svm.SVC` mit dem linearen Kernel (`kernel='linear'`) und trainieren Sie diese auf den Features `petal length (cm)` und `petal width (cm)` vom `Train-Set` (`X_train`, `y_train`).
2. Predicten (`clf.predict`) Sie die Vorhersagen auf dem `Validation-Set` (`X_val`). Hier müssen Sie wieder die entsprechenden Features selektieren.
3. Messen Sie die Genauigkeit (englisch Accuracy) mittels `sklearn.metrics.accuracy_score` von unseren Vorhersagen `y_val_hat` und den tatsächlichen Labels `y_val`.
4. Visualisieren Sie die decision boundary von unserem Modell mit der Hilfe von der oben definierten `plot_decision_regions` Funktion. Was zeigt dieser Plot?


In [ ]:
# TODO


### (Extra) Aufgabe 4.3.3

Wir haben in Aufgabe 2.2 und Aufgabe 4.3.2 die `Decision Regions` der Logistischen Regression und der Support Vector Machine visualisiert.
Das sollte so aussehen:

Logistische Regression     |  Support Vector Machine
:-------------------------:|:-------------------------:
![Logistische Regression oder SVM?](img/lr.png) |  ![Logistische Regression oder SVM?](img/svm.png)

1. Erkläre den Unterschied der Decision Boundary mit Ihrem Wissen darüber wie diese Modelle funktionieren und trainiert werden. Woran man das zugrundeliegende Modell anhand der `Decision Regions` erkennen?


In [ ]:
# TODO

### (Extra) Aufgabe 4.3.4

Die Support Vector Machine kann mittels `kernel` eine nicht lineares Modell werden. Der `Kernel` beschreibt dabei ein indirektes Feature Engineering und kann effizienter sein als explizites Feature Engineering. Generell kann man diesen Trick in verschiedenen Modellen einsetzen (Lineare Regression, Logistische Regression, KNN, ...), er ist jedoch nur bei `Support Vector Machine` out-of-the-box in sklearn verfügbar, da er in der SVM effizient(er) ist während der Anwendung als bei anderen Modellen.

Welcher Kernel wir verwenden und welche Parameter wir für einen spezifischen Kernel verwenden sind `Hyper-Parameter` vom `Support Vector Machine` Modell.
Hier verwenden wir den `rbf` Kernel mit default `gamma` Wert.

1. Erstellen Sie eine `sklearn.svm.SVC` mit dem `rbf` Kernel (`kernel='rbf'` und `gamma=1`) und trainieren Sie diese auf den Features `petal length (cm)` und `petal width (cm)` vom `Train-Set` (`X_train`, `y_train`).
2. Predicten (`clf.predict`) Sie die Vorhersagen auf dem `Validation-Set` (`X_val`). Hier müssen Sie wieder die entsprechenden Features selektieren.
3. Messen Sie die `Accuracy` mittels `sklearn.metrics.accuracy_score` von unseren Vorhersagen `y_val_hat` und den tatsächlichen Labels `y_val`.
4. Visualisieren Sie die `Decision Boundary` von unserem Modell mit der Hilfe von der oben definierten `plot_decision_regions` Funktion.
5. Spielen Sie mit dem `gamma` Parameter. Setzen Sie ihn auf `0.1` oder auf `10`. Was verändert sich?

In [ ]:
# TODO

## Schlusswort Aufgabenblatt 3

Im Aufgabenblatt 3 haben wir die `Classification` genauer angeschaut.
Wir haben verschiedene Plots für die `Datenanalyse`, verschiedene `Metriken` und verschiedene Modelle angeschaut.
Es ist vorallem wichtig, dass Sie ein erstes Gefühl für die `Classification` bekommen haben und die Unterschiede zur Regression (Aufgabenblatt `linear_regression.ipynb`) klar verstehen.

Der verwendete Iris-Datensatz ist ein oft verwendeter erster Datensatz. In der Praxis hat man oft schwierigere Datensätze, dass man eine Genauigkeit von 100% erreicht ist eher unüblich und je nach Problem unmöglich.